## ROUGE score

In [ ]:
import pandas as pd
from IPython.display import display

import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
file_path = filedialog.askopenfilename(title="Select  CSV file")
df = pd.read_csv(file_path)

display(df.head())
def rouge1_precision(reference,candidate):
    """
    ROUGE-1 Precision: How many words in candidate also appear in reference, divided by candidate's length
    """
    cand_tokens = set(candidate.lower().split())
    ref_tokens = set(reference.lower().split())
    if len(cand_tokens) == 0:
        return 0.0
    overlap = cand_tokens & ref_tokens
    return len(overlap) / len(cand_tokens)


df['rouge1_precision'] = df.apply(
    lambda row: rouge1_precision(str(row['First_attempt']), str(row['last_attempt'])),
    axis=1
)

df.to_csv("ROUGE_Result.csv", index=False)



## BERTScore

In [ ]:
import pandas as pd
from IPython.display import display
import random
from bert_score import score as bert_score
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
file_path = filedialog.askopenfilename(title="Select  CSV file")
df = pd.read_csv(file_path)


candidates = df['First_attempt'].tolist()
references = df['last_attempt'].tolist()
rows = df['row'].tolist()

tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
max_len = 510

def safe_truncate(text, tokenizer, max_len=510):
    tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(tokens) > max_len:
        # Remove special tokens before truncation if present
        tokens = tokens[:max_len]
        return tokenizer.decode(tokens, skip_special_tokens=True)
    else:
        return text
    
def safe_cast_to_str(x):
    if x is None:
        return ""
    try:
        s = str(x)
        if s.lower() == 'nan':
            return ""
        return s
    except Exception:
        return ""

candidates_trunc = [safe_truncate(safe_cast_to_str(t), tokenizer, max_len) for t in candidates]
references_trunc = [safe_truncate(safe_cast_to_str(t), tokenizer, max_len) for t in references]

P= bert_score(candidates_trunc, references_trunc, lang='en', model_type="emilyalsentzer/Bio_ClinicalBERT", num_layers=12)

for i, (p) in enumerate(zip(P)):
    print(f"Row {i}: Precision={p.item():.3f}")

bert_results = pd.DataFrame({
    'row': rows,
    'BERTScore_Precision': [p.item() for p in P],

})

# Save to CSV
bert_results.to_csv("BERTScore_Result.csv", index=False)




# Tailored Metric

In [ ]:
!pip -q install "transformers>=4.41,<5" "torch>=2.3,<3"

import re, csv, math
from dataclasses import dataclass, field
from typing import Dict, Tuple, Optional, List, Any
from collections import defaultdict, Counter
from transformers import pipeline

EXPAND_BILATERAL = True
TRACE = False

MODEL_NAME = "d4data/biomedical-ner-all"
NER_GATE_PRED = True
NER_GATE_GT = False

MERGE_UNPAIRED_INTO_MISMATCH = True
MERGE_MISSING_METRIC_INTO_MISMATCH = False

NULL_LIKE = {"", "n/a", "na", "not available", "none", "null", "-", "—"}

#Metric aliases
def _mk_aliases():
    aliases = {}
    def add(canon, *variants):
        for v in variants: aliases[v] = canon
    add("volume_w_score",
        "volume_w_score","volume w score","volume_wscore","volume wscore",
        "volume_w","volume score","volume","vol","vol score","vol w score","vol_w_score")
    add("relevance_w_score",
        "relevance_w_score","relevance w score","relevance_wscore","relevance wscore",
        "relevance","rel","rel score")
    add("cortical_thickness_w_score",
        "cortical_thickness_w_score","cortical thickness w score",
        "cortical_thickness_wscore","cortical thickness wscore",
        "cortical_thickness","cortical thickness","thickness","thickness score")
    return aliases
METRIC_ALIASES = _mk_aliases()

# Laterality / patterns
LATS_MAP = {"left":"LEFT","l":"LEFT","lt":"LEFT",
            "right":"RIGHT","r":"RIGHT","rt":"RIGHT",
            "bilateral":"BILATERAL","both":"BILATERAL"}

LAT_REGION_RE = re.compile(
    r"\b(?P<lat>Left|Right|Bilateral|L|R|Lt|Rt)\b\s+(?P<region>[^:,(]+?)\s*(?=[:(,]|$)",
    re.IGNORECASE
)
NEAREST_PARENS_RE = re.compile(r"^\s*[:;,]?\s*\(([^)]{0,400})\)")

## Atrophy / Severity 
ATROPHY_POS_RE = re.compile(r"\batroph(?:y|ied|ic)\b|volume\s+loss|reduced\s+volume|shrinkage", re.IGNORECASE)
ATROPHY_NEG_RE = re.compile(r"\bno\s+atrophy\b|\bwithout\s+atrophy\b", re.IGNORECASE)
STRONG_BEFORE_PATH_RE   = re.compile(r"\b(strong|severe|severely|marked|pronounced|advanced)\b(?:\s+\w+){0,3}\s+patholog(?:y|ies)\b", re.IGNORECASE)
MODERATE_BEFORE_PATH_RE = re.compile(r"\b(moderate|moderately|intermediate)\b(?:\s+\w+){0,3}\s+patholog(?:y|ies)\b", re.IGNORECASE)
MILD_BEFORE_PATH_RE     = re.compile(r"\b(mild|mildly|slight|slightly|minimal|minimally|trace)\b(?:\s+\w+){0,3}\s+patholog(?:y|ies)\b", re.IGNORECASE)

# Global NO-PATHOLOGY equivalents
NO_PATHOLOGY_RE = re.compile(
    "|".join([
        r"\bno\s+(?:significant\s+)?patholog(?:y|ies)\b",
        r"\bno\s+regions?\s+(?:show|shows|demonstrate|demonstrates)\s+(?:any\s+)?(?:significant\s+)?pathology\b",
        r"\bno\s+abnormalit(?:y|ies)\b",
        r"\bno\s+abnormality\s+detected\b",
        r"\bno\s+evidence\s+of\s+(?:significant\s+)?(?:pathology|abnormalit(?:y|ies))\b",
        r"\bwithin\s+(?:normal|expected)\s+limits\b",
        r"\bunremarkable\b",
        r"\bnormal\s+(?:study|scan|exam|finding|findings)\b",
        r"\bno\s+pathology\s+for\s+this\s+patient\b",
    ]), re.IGNORECASE
)

def is_no_pathology(text: str) -> bool:
    return bool(NO_PATHOLOGY_RE.search(text or ""))

def _normalize_punct(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    return (s.replace("\u2212","-")
             .replace("\u2010","-").replace("\u2011","-").replace("\u2012","-")
             .replace("\u2013","-").replace("\u2014","-")
             .replace("\u00A0"," "))

def norm_null(x: Optional[str]) -> Optional[str]:
    if x is None: return None
    s = _normalize_punct(str(x)).strip().lower()
    return None if s in NULL_LIKE else _normalize_punct(str(x)).strip()

_num_re = re.compile(r"^[\+\-]?\d+(?:\.\d+)?(?:[eE][\+\-]?\d+)?$")
def as_number_or_none(x: Optional[str]) -> Optional[float]:
    sx = norm_null(x)
    if sx is None: return None
    s = _normalize_punct(sx).replace(",", "").strip()
    if _num_re.match(s):
        try: return float(s)
        except ValueError: return None
    m = re.findall(r"[\+\-]?\d+(?:\.\d+)?(?:[eE][\+\-]?\d+)?", s)
    if m:
        try: return float(m[-1])
        except ValueError: return None
    return None

def norm_key(k: str) -> str:
    k = _normalize_punct(k).strip().lower()
    k = k.replace("-", " ").replace("/", " ")
    k = "_".join(k.split())
    return METRIC_ALIASES.get(k, k)

def canonical_region(s: str) -> str:
    return " ".join(_normalize_punct(s).strip().lower().split())

def _expand_key(key: Tuple[str,str]) -> List[Tuple[str,str]]:
    reg, lat = key
    if lat == "BILATERAL" and EXPAND_BILATERAL:
        return [(reg, "LEFT"), (reg, "RIGHT")]
    return [key]

def _sentence_bounds(text: str, start_idx: int, end_idx: int) -> Tuple[int, int]:
    left_delims = [text.rfind(d, 0, start_idx) for d in ('.',';','\n')]
    left = max([i for i in left_delims if i != -1], default=-1) + 1
    rights = [text.find(d, end_idx) for d in ('.',';','\n')]
    rights = [i for i in rights if i != -1]
    right = min(rights) if rights else len(text)
    return left, right

def parse_metrics(block: Optional[str]) -> Dict[str, Optional[Any]]:
    metrics: Dict[str, Optional[Any]] = {}
    if not block: return metrics
    items = [p.strip() for p in block.split(",") if p.strip()]
    for item in items:
        if ":" not in item:
            metrics[norm_key(item)] = None
            continue
        k, v = item.split(":", 1)
        k = norm_key(k); v = v.strip()
        if norm_null(v) is None:
            metrics[k] = None
        else:
            num = as_number_or_none(v)
            metrics[k] = num if num is not None else _normalize_punct(v.strip())
    return metrics

ner = pipeline("token-classification", model=MODEL_NAME, aggregation_strategy="simple")

def ner_spans(text: str) -> List[Tuple[int,int]]:
    if not text:
        return []
    spans = []
    for ent in ner(text):
        spans.append((int(ent.get("start", 0)), int(ent.get("end", 0))))
    return spans

def overlaps(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    return not (a_end <= b_start or b_end <= a_start)

@dataclass
class Finding:
    region: str
    laterality: str
    metrics: Dict[str, Optional[Any]] = field(default_factory=dict)
    atrophy: bool = False
    severity: Optional[str] = None

def parse_all_occurrences(text: str, ner_gate: bool = False):
    by_key: Dict[Tuple[str,str], List[Finding]] = defaultdict(list)
    raw_counts: Counter = Counter()
    if not text: return by_key, raw_counts

    spans = ner_spans(text) if ner_gate else []

    for m in LAT_REGION_RE.finditer(text):
        lat_raw = m.group("lat").lower()
        lat = LATS_MAP.get(lat_raw, lat_raw.upper())
        reg_txt = canonical_region(m.group("region"))

        if ner_gate:
            reg_start = m.start("region")
            reg_end   = m.end("region")
            ok = any(overlaps(reg_start, reg_end, s, e) for (s, e) in spans)
            if not ok:
                continue

        s_left, s_right = _sentence_bounds(text, m.start(), m.end())
        scope_text = text[s_left:s_right]

        tail = text[m.end(): m.end()+400]
        pm = NEAREST_PARENS_RE.search(tail)
        metrics = parse_metrics(pm.group(1)) if pm else {}

        if ATROPHY_NEG_RE.search(scope_text): atrophy_flag = False
        else: atrophy_flag = bool(ATROPHY_POS_RE.search(scope_text))

        sev = None
        if STRONG_BEFORE_PATH_RE.search(scope_text):   sev = "STRONG"
        elif MODERATE_BEFORE_PATH_RE.search(scope_text): sev = "MODERATE"
        elif MILD_BEFORE_PATH_RE.search(scope_text):     sev = "MILD"

        for k in _expand_key((reg_txt, lat)):
            by_key[k].append(Finding(k[0], k[1], metrics, atrophy_flag, sev))
            raw_counts[k] += 1

    if TRACE:
        print("[PARSE] occurrences:")
        for k, lst in by_key.items():
            for i, f in enumerate(lst, 1):
                print(f"  {k} occ {i} -> metrics={f.metrics} | atrophy={f.atrophy} | severity={f.severity}")
    return by_key, raw_counts

def trunc1(x: float) -> float:
    return math.trunc(x*10.0)/10.0

def numeric_equal_1dec_trunc(vt: Any, vp: Any) -> Optional[bool]:
    def as_num(v):
        n = v if isinstance(v, (int,float)) else as_number_or_none(v)
        return float(n) if isinstance(n, (int,float)) and math.isfinite(n) else None
    t = as_num(vt); p = as_num(vp)
    if t is None or p is None: return None
    return trunc1(t) == trunc1(p)

#per-row errors

@dataclass
class RowErrorCounts:
    region_missing: int
    region_extra: int
    region_errors: int
    v_mismatch_occ: int
    v_missing_metric_occ: int
    v_unpaired_occ_gt: int
    v_unpaired_occ_pr: int
    rep_mismatch: int
    rep_missing_metric: int
    rep_unpaired: int
    value_errors_total: int
    atrophy_errors: int
    severity_errors: int
    total_errors: int
    details: List[str] = field(default_factory=list)

def count_errors(gt: str, pr: str) -> RowErrorCounts:
    details: List[str] = []

    if is_no_pathology(gt) and is_no_pathology(pr):
        return RowErrorCounts(0,0,0, 0,0,0,0, 0,0,0, 0, 0,0, 0, details)

    gt_occ, gt_counts = parse_all_occurrences(gt or "", ner_gate=NER_GATE_GT)
    pr_occ, pr_counts = parse_all_occurrences(pr or "", ner_gate=NER_GATE_PRED)

    if not gt_counts and not pr_counts:
        return RowErrorCounts(0,0,0, 0,0,0,0, 0,0,0, 0, 0,0, 0, details)

    all_keys = set(gt_counts) | set(pr_counts)
    region_missing = region_extra = 0
    miss_list = []; extra_list = []
    for k in sorted(all_keys):
        ct = gt_counts.get(k, 0); cp = pr_counts.get(k, 0)
        if ct > cp:
            d = ct - cp; region_missing += d; miss_list.append((k, d))
        elif cp > ct:
            d = cp - ct; region_extra += d; extra_list.append((k, d))
    region_errors = region_missing + region_extra
    if miss_list:  details.append(f"[REGION] Missing: {miss_list}")
    if extra_list: details.append(f"[REGION] Extra: {extra_list}")

    v_mismatch_occ = 0
    v_missing_metric_occ = 0
    v_unpaired_occ_gt = 0
    v_unpaired_occ_pr = 0

    keys_union = sorted(set(gt_occ) | set(pr_occ))
    for key in keys_union:
        gt_list = gt_occ.get(key, [])
        pr_list = pr_occ.get(key, [])
        m = min(len(gt_list), len(pr_list))

        for i in range(m):
            g = gt_list[i]; p = pr_list[i]
            has_numeric_or_text_diff = False
            has_missing_metric = False

            all_metrics = set(g.metrics.keys()) | set(p.metrics.keys())
            for mk in all_metrics:
                g_val = g.metrics.get(mk); p_val = p.metrics.get(mk)
                g_missing = (norm_null(g_val) is None)
                p_missing = (norm_null(p_val) is None)

                if g_missing and p_missing:
                    continue

                eq_num = numeric_equal_1dec_trunc(g_val, p_val)
                if eq_num is True:
                    continue
                if eq_num is False:
                    has_numeric_or_text_diff = True
                    break

                if g_missing != p_missing:
                    has_missing_metric = True
                elif str(g_val).strip().lower() != str(p_val).strip().lower():
                    has_numeric_or_text_diff = True
                    break

            if has_numeric_or_text_diff:
                v_mismatch_occ += 1
                details.append(f"[VALUE_MISMATCH] {key} occ {i+1}")
            elif has_missing_metric:
                v_missing_metric_occ += 1
                details.append(f"[VALUE_MISSING_METRIC] {key} occ {i+1}")

        def occ_has_value(f):
            return any(norm_null(v) is not None for v in f.metrics.values())

        for f in gt_list[m:]:
            if occ_has_value(f):
                v_unpaired_occ_gt += 1
                details.append(f"[VALUE_UNPAIRED_GT] {key} extra GT occ")
        for f in pr_list[m:]:
            if occ_has_value(f):
                v_unpaired_occ_pr += 1
                details.append(f"[VALUE_UNPAIRED_PR] {key} extra PR occ")

    #Report Numbers
    rep_mismatch = v_mismatch_occ
    rep_missing_metric = v_missing_metric_occ
    rep_unpaired = v_unpaired_occ_gt + v_unpaired_occ_pr

    if MERGE_UNPAIRED_INTO_MISMATCH:
        rep_mismatch += rep_unpaired
        rep_unpaired = 0
    if MERGE_MISSING_METRIC_INTO_MISMATCH:
        rep_mismatch += rep_missing_metric
        rep_missing_metric = 0

    value_errors_total = rep_mismatch + rep_missing_metric + rep_unpaired

    atrophy_errors = 0
    severity_errors = 0
    for key in sorted(set(gt_occ) & set(pr_occ)):
        gt_list = gt_occ[key]; pr_list = pr_occ[key]
        m = min(len(gt_list), len(pr_list))
        for i in range(m):
            g = gt_list[i]; p = pr_list[i]
            if g.atrophy != p.atrophy:
                atrophy_errors += 1
                details.append(f"[ATROPHY] {key} occ {i+1}: GT={g.atrophy}, PR={p.atrophy}")
            if (g.severity or p.severity) and (g.severity != p.severity):
                severity_errors += 1
                details.append(f"[SEVERITY] {key} occ {i+1}: GT={g.severity}, PR={p.severity}")

    total_errors = region_errors + value_errors_total + atrophy_errors + severity_errors

    return RowErrorCounts(
        region_missing=region_missing, region_extra=region_extra, region_errors=region_errors,
        v_mismatch_occ=v_mismatch_occ, v_missing_metric_occ=v_missing_metric_occ,
        v_unpaired_occ_gt=v_unpaired_occ_gt, v_unpaired_occ_pr=v_unpaired_occ_pr,
        rep_mismatch=rep_mismatch, rep_missing_metric=rep_missing_metric, rep_unpaired=rep_unpaired,
        value_errors_total=value_errors_total,
        atrophy_errors=atrophy_errors, severity_errors=severity_errors,
        total_errors=total_errors, details=details
    )

def _detect_delimiter(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8") as f:
            sample = f.read(4096)
        import csv as _csv
        dialect = _csv.Sniffer().sniff(sample, delimiters=",;\t|")
        return dialect.delimiter
    except Exception:
        return ","

def eval_csv_full(path: str):
    delim = _detect_delimiter(path)
    nrows = 0

    agg_region_missing = agg_region_extra = agg_region_errors = 0
    agg_rep_mismatch = agg_rep_missing_metric = agg_rep_unpaired = 0
    agg_val_total = 0
    agg_atrophy = agg_severity = 0
    agg_total = 0
    rows_with_any_error = 0

    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter=delim)
        if not reader.fieldnames:
            print("[ERROR] CSV has no header row. Expected: ground_truth,llm_output"); return
        name_map = {n.lower(): n for n in reader.fieldnames if isinstance(n, str)}
        for needed in ("ground_truth","llm_output"):
            if needed not in name_map:
                print(f"[ERROR] Missing header: {needed}. Found: {reader.fieldnames}")
                return
        gt_col, llm_col = name_map["ground_truth"], name_map["llm_output"]

        for row in reader:
            nrows += 1
            gt = (row.get(gt_col) or "").strip()
            pr = (row.get(llm_col) or "").strip()

            rc = count_errors(gt, pr)

            agg_region_missing += rc.region_missing
            agg_region_extra   += rc.region_extra
            agg_region_errors  += rc.region_errors

            agg_rep_mismatch       += rc.rep_mismatch
            agg_rep_missing_metric += rc.rep_missing_metric
            agg_rep_unpaired       += rc.rep_unpaired
            agg_val_total          += rc.value_errors_total

            agg_atrophy        += rc.atrophy_errors
            agg_severity       += rc.severity_errors
            agg_total          += rc.total_errors

            if rc.total_errors > 0: rows_with_any_error += 1

            print(
                f"Row {nrows}: "
                f"region_err={rc.region_errors} (missing={rc.region_missing}, extra={rc.region_extra}) | "
                f"value_total={rc.value_errors_total} "
                f"[mismatch={rc.rep_mismatch}, missing_metric={rc.rep_missing_metric}, unpaired={rc.rep_unpaired}] | "
                f"atrophy={rc.atrophy_errors} | severity={rc.severity_errors} | "
                f"TOTAL={rc.total_errors}"
            )
            for d in rc.details:
                print("  -", d)

    if nrows == 0:
        print("[INFO] No rows."); return

    correct_rows = nrows - rows_with_any_error

    print("\n=== SUMMARY ===")
    print(f"Rows: {nrows} | Rows with any error: {rows_with_any_error}/{nrows} "
          f"({rows_with_any_error*100.0/nrows:.1f}%)")
    print(f"Correct rows (exact match): {correct_rows}/{nrows} "
          f"({correct_rows*100.0/nrows:.1f}%)")
    print(f"Region errors:      {agg_region_errors}  (missing={agg_region_missing}, extra={agg_region_extra})")
    print(f"Value errors total: {agg_val_total}  "
          f"(mismatch={agg_rep_mismatch}, missing_metric={agg_rep_missing_metric}, unpaired={agg_rep_unpaired})")
    print(f"Atrophy errors:     {agg_atrophy}")
    print(f"Severity errors:    {agg_severity}")
    print(f"TOTAL errors:       {agg_total}")


#Run
def _run():
    try:
        from google.colab import files  # type: ignore
        print("Upload CSV (headers: ground_truth,llm_output)...")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded."); return
        path = next(iter(uploaded))
        print(f"Using: {path}")
        eval_csv_full(path)
    except ModuleNotFoundError:
        print("[INFO] Not in Colab. Set CSV_PATH below and run again.")

_run()